In [1]:
import json
import pandas as pd
import os
from unidecode import unidecode

In [2]:
JV_REPLACE=True
def jv_replace(text: str):
    text = text.replace('j', 'i')
    text = text.replace('J', 'I')
    text = text.replace('v', 'u')
    text = text.replace('V', 'U')
    return text

In [3]:
data_dir = '../data/final_dataset'
long_ans_files = ['certamen_translation_long.json', 
                  'junior_scholarship_translation_long.json'
                  ]
long_ans_files = [os.path.join(data_dir, f) for f in long_ans_files]

file_to_data = {}
for file in long_ans_files:
    base_name = os.path.basename(file)
    file_to_data[base_name] = []
    with open(file, 'r') as f:
        file_to_data[base_name] += json.load(f)

    print(base_name, len(file_to_data[base_name]))

certamen_translation_long.json 916
junior_scholarship_translation_long.json 350


In [4]:
file_to_resp = {}
#model_name = 'llama3-turbo'
#model_name = 'qwq'
model_name = 'o3-mini'

model_resp_dir = f'../data/model_responses/{model_name}'

for file in file_to_data.keys():
    base_name = os.path.basename(file)
    file_to_resp[base_name] = {}
    with open(os.path.join(model_resp_dir, base_name), 'r') as f:
        file_to_resp[base_name] = json.load(f)
    print(base_name, len(file_to_resp[base_name]))


certamen_translation_long.json 916
junior_scholarship_translation_long.json 350


In [5]:
# make dfs
question_dfs = []
for file in file_to_data.keys():
    questions = file_to_data[file]
    question_df = pd.DataFrame(questions)
    question_dfs.append(question_df)

In [6]:
for i, file in enumerate(file_to_data.keys()):
    question_df = question_dfs[i]
    answer_dict = file_to_resp[file]
    answer_col = []
    for (i, row) in question_df.iterrows():
        q_id = row['question_id']

        model_answer = answer_dict[q_id]
        answer_col.append(model_answer)

    question_df['raw_resp'] = answer_col

In [7]:
def parse_resp(resp_text):
    '''get the answer from the full response. usually on the last line as Answer: answer'''

    lines = resp_text.split('\n')
    lines = [line.replace('*', '') for line in lines]
    # usually in last line
    line = lines[-1]
    #line = line.replace('*', '')
    # look for "Answer: A"
    if 'Answer:' in line:
        answer =line.split('Answer:')[1].strip()
        return answer
    else:
        for i in range(len(lines)-1, 0, -1):
            line = lines[i]
            if 'Answer:' in line:
                try:
                    answer =line.split('Answer: ')[1].strip()
                    # add remaining lines until reaching a line starting with "Explanation"
                    for j in range(i+1, len(lines)):
                        next_line = lines[j]
                        if "Explanation" in next_line:
                            return answer
                        answer += '\n'+next_line
                except Exception as e:
                    print(resp_text)
                    print('line:', line)
                    print(e)
                
                return answer
        print('error parsing')
        print(resp_text)
        return ''

In [8]:
for i, file in enumerate(file_to_data.keys()):
    question_df = question_dfs[i]
    question_df['parsed_resp'] = question_df['raw_resp'].apply(parse_resp)
    question_dfs[i] = question_df
question_dfs[0]

,source_name,source_year,question_id,question_format,question_content,difficulty,question_language,answer_language,question,multiple_choice_options,answers,constrained,raw_resp,parsed_resp
0,NJCL-Certamen,1996,NJCL-Certamen_1996_8a_0,short_answer,translation,unknown,english,english,Translate the motto of Alabama: Audēmus iūra n...,[],"[WE DARE TO DEFEND OUR RIGHTS, WE DARE TO DEFE...",False,"The motto ""Audēmus iūra nostra dēfendere"" tran...",We dare defend our rights.
1,NJCL-Certamen,1996,NJCL-Certamen_1996_8b_0,short_answer,translation,unknown,english,english,Translate the motto of the Dominion of Canada:...,[],[FROM SEA TO SEA],False,"The motto ā mari usque ad mare means ""From sea...",From sea to sea.
2,NJCL-Certamen,1996,NJCL-Certamen_1996_8c_0,short_answer,translation,unknown,english,english,Translate the motto of Wellesley College: nōn ...,[],"[NOT TO BE SERVED, NOT TO BE MANAGED, BUT TO S...",False,The motto is typically rendered in English as ...,"Not to be served, but to serve."
3,NJCL-Certamen,1996,NJCL-Certamen_1996_15a_3,short_answer,translation,unknown,english,english,Translate into English the following maxim of ...,[],"[If you want peace, prepare for war]",False,"The maxim translates to: ""If you want peace, p...","If you want peace, prepare for war."
4,NJCL-Certamen,1996,NJCL-Certamen_1996_15b_3,short_answer,translation,unknown,english,english,Translate into English the following quotation...,[],"[Divine nature gave us the fields, human art b...",False,Divine nature gave the fields; human art built...,Divine nature gave the fields; human art built...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
911,NJCL-Certamen,2002,NJCL-Certamen_2002_B1_865,short_answer,translation,advanced,english,english,Translate the following sentence into English:...,[],"[IT IS NOT FITTING FOR US TO WORK THIS SUMMER,...",False,It is not to our advantage for us to work this...,It is not to our advantage for us to work this...
912,NJCL-Certamen,2002,NJCL-Certamen_2002_B2_874,short_answer,translation,advanced,english,english,Translate the following sentence into English:...,[],[IT IS BECOMING FOR ALL TO VISIT TEMPLES OF TH...,False,"The sentence translates to: ""It is proper for ...",It is proper for everyone to visit the temples...
913,NJCL-Certamen,2002,NJCL-Certamen_2002_13_77,short_answer,translation,advanced,english,english,Translate the following sentence into English:...,[],[My brother saw the dog wounded by a rock],False,"The sentence translates as: ""My brother saw th...",My brother saw the dog wounded by a stone.
914,NJCL-Certamen,2002,NJCL-Certamen_2002_13a_71,short_answer,translation,advanced,english,english,Translate the following sentence into English:...,[],"[Having seen the wounded dog, my brother wante...",False,The Latin sentence reads:\n\n cane vulnerātō ...,"When the wounded dog was seen, my brother imme..."


In [9]:
def get_refs_and_responses(df: pd.DataFrame, ignore_macrons: bool):
    refs = [] # list of lists of strings
    model_responses = []

    for i, row in df.iterrows():
        answer_lang = row['answer_language']
        this_refs = row['answers']
        this_resp = row['parsed_resp']

        # jv replace the resp and refs
        if answer_lang == 'latin' and JV_REPLACE:
            this_refs = [jv_replace(text) for text in this_refs]
            this_resp = jv_replace(this_resp)
        
        if answer_lang == 'latin' and ignore_macrons:
            this_refs = [unidecode(text) for text in this_refs]
            this_resp = unidecode(this_resp)

        refs.append(this_refs)
        model_responses.append(this_resp)

    # make sure length of each ref list is the same
    # what is the max len?
    max_ref = max([len(r) for r in refs])
    for i, r in enumerate(refs):
        this_len = len(r)
        n_to_add = max_ref - this_len 

        r += [''] * n_to_add

    # transpose the refs
    new_refs = [['' for _ in range(len(refs))] for _ in range(max_ref)] # max ref x number of sentences
    for j, ref_list in enumerate(refs): # j is as long as number of sentences 
        for i, ref in enumerate(ref_list): # i is as long as number of refs
            new_refs[i][j] = ref
    
    return new_refs, model_responses

In [10]:
# create combined df, with new columns called "source file"
for i, file in enumerate(file_to_data.keys()):
    question_df = question_dfs[i]
    question_df['source_file'] = [file] * len(question_df)
    question_dfs[i] = question_df

combined_df = pd.concat(question_dfs)
combined_df

,source_name,source_year,question_id,question_format,question_content,difficulty,question_language,answer_language,question,multiple_choice_options,answers,constrained,raw_resp,parsed_resp,source_file,question_text
0,NJCL-Certamen,1996,NJCL-Certamen_1996_8a_0,short_answer,translation,unknown,english,english,Translate the motto of Alabama: Audēmus iūra n...,[],"[WE DARE TO DEFEND OUR RIGHTS, WE DARE TO DEFE...",False,"The motto ""Audēmus iūra nostra dēfendere"" tran...",We dare defend our rights.,certamen_translation_long.json,NaN
1,NJCL-Certamen,1996,NJCL-Certamen_1996_8b_0,short_answer,translation,unknown,english,english,Translate the motto of the Dominion of Canada:...,[],[FROM SEA TO SEA],False,"The motto ā mari usque ad mare means ""From sea...",From sea to sea.,certamen_translation_long.json,NaN
2,NJCL-Certamen,1996,NJCL-Certamen_1996_8c_0,short_answer,translation,unknown,english,english,Translate the motto of Wellesley College: nōn ...,[],"[NOT TO BE SERVED, NOT TO BE MANAGED, BUT TO S...",False,The motto is typically rendered in English as ...,"Not to be served, but to serve.",certamen_translation_long.json,NaN
3,NJCL-Certamen,1996,NJCL-Certamen_1996_15a_3,short_answer,translation,unknown,english,english,Translate into English the following maxim of ...,[],"[If you want peace, prepare for war]",False,"The maxim translates to: ""If you want peace, p...","If you want peace, prepare for war.",certamen_translation_long.json,NaN
4,NJCL-Certamen,1996,NJCL-Certamen_1996_15b_3,short_answer,translation,unknown,english,english,Translate into English the following quotation...,[],"[Divine nature gave us the fields, human art b...",False,Divine nature gave the fields; human art built...,Divine nature gave the fields; human art built...,certamen_translation_long.json,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
345,latin-grammar-junior-scholarship,1884,latin-grammar-junior-scholarship_114.7.1,short_answer,translation,unknown,english,english,NaN,[],"[The more a man denies himself, the more will ...",True,The more things each person has denied for him...,The more things each person has denied for him...,junior_scholarship_translation_long.json,Translate the following Latin phrase into Engl...
346,latin-grammar-junior-scholarship,1884,latin-grammar-junior-scholarship_114.8.2,short_answer,translation,unknown,english,latin,NaN,[],[Extremo tertio libro.],False,Ad finem libri tertii.\n\nAnswer: Ad finem lib...,Ad finem libri tertii.,junior_scholarship_translation_long.json,"Put into Latin—""At the end of the third book."""
347,latin-grammar-junior-scholarship,1884,latin-grammar-junior-scholarship_114.8.4,short_answer,translation,unknown,english,latin,NaN,[],[Alii alia putant.],False,"Aliqui unum putant, alii alterum.\n\nAnswer: A...","Aliqui unum putant, alii alterum.",junior_scholarship_translation_long.json,"Put into Latin—""Some think one thing, some ano..."
348,latin-grammar-junior-scholarship,1884,latin-grammar-junior-scholarship_app.V.4.1,short_answer,translation,unknown,english,english,NaN,[],[Let war yield to peace.],False,"The phrase ""Cedant arma togae"" is typically tr...",Let arms yield to the toga.,junior_scholarship_translation_long.json,Translate —Cedant arma togae.


In [11]:
# save combined df
save_dir = f'../data/model_responses_parsed/{model_name}'
if not os.path.exists(save_dir):
    os.makedirs(save_dir)
# drop the columns question, multiple_choice_options, n_required, correctness_logic
save_df = combined_df.drop(columns=['question', 'question_text', 'multiple_choice_options'])
save_dict = save_df.to_dict(orient='records')



with open(os.path.join(save_dir, 'translation_long.json'), 'w') as f:
    json.dump(save_dict, f, indent=4)

In [12]:
# latin answer
latin_ans_df = combined_df[combined_df['answer_language'] == 'latin']
lat_refs, lat_sys = get_refs_and_responses(latin_ans_df, ignore_macrons=False)
lat_refs_no_macrons, lat_sys_no_macrons = get_refs_and_responses(latin_ans_df, ignore_macrons=True)

In [13]:
# lat constrained and unconstrained
lat_const_df = latin_ans_df[latin_ans_df['constrained'] == True]
lat_const_refs, lat_const_sys = get_refs_and_responses(lat_const_df, ignore_macrons=False)
lat_const_refs_no_mac, lat_const_sys_no_mac = get_refs_and_responses(lat_const_df, ignore_macrons=True)

lat_unconst_df = latin_ans_df[latin_ans_df['constrained'] == False]
lat_unconst_refs, lat_unconst_sys = get_refs_and_responses(lat_unconst_df, ignore_macrons=False)
lat_unconst_refs_no_mac, lat_unconst_sys_no_mac = get_refs_and_responses(lat_unconst_df, ignore_macrons=True)


In [15]:
# english answer
eng_ans_df =  combined_df[combined_df['answer_language'] == 'english']
eng_refs, eng_sys = get_refs_and_responses(eng_ans_df, ignore_macrons=False)

In [16]:
# eng constrained and unconstrained
eng_const_df = eng_ans_df[eng_ans_df['constrained'] == True]
eng_const_refs, eng_const_sys = get_refs_and_responses(eng_const_df, ignore_macrons=False)

eng_unconst_df =  eng_ans_df[eng_ans_df['constrained'] == False]
eng_unconst_refs, eng_unconst_sys = get_refs_and_responses(eng_unconst_df, ignore_macrons=False)

In [17]:
latin_ans_df

,source_name,source_year,question_id,question_format,question_content,difficulty,question_language,answer_language,question,multiple_choice_options,answers,constrained,raw_resp,parsed_resp,source_file,question_text
19,NJCL-Certamen,1996,NJCL-Certamen_1996_1a_11,short_answer,translation,unknown,english,latin,Say in Latin : Where are the horses of the mas...,[],"[UBI SUNT DOMINĪ CABALLĪ?, UBI SUNT MAGISTRĪ C...",False,Ubi sunt equi domini?\n\nAnswer: Ubi sunt equi...,Ubi sunt equi domini?,certamen_translation_long.json,NaN
20,NJCL-Certamen,1996,NJCL-Certamen_1996_1b_11,short_answer,translation,unknown,english,latin,Say in Latin: The boys do not want to stay at ...,[],"[PUERĪ DOMĪ MANĒRE NŌN CUPIUNT, PUERĪ DOMĪ MAN...",False,"In Latin, you would say: Pueri non volunt domi...",Pueri non volunt domi manere.,certamen_translation_long.json,NaN
21,NJCL-Certamen,1996,NJCL-Certamen_1996_1c_11,short_answer,translation,unknown,english,latin,Say in Latin: My friend was captured by pirates.,[],"[MEUS AMICUS Ā PĪRĀTĪS CAPTUS EST, MEA AMICA Ā...",False,Amicus meus a piratis captus est.\nAnswer: Ami...,Amicus meus a piratis captus est.,certamen_translation_long.json,NaN
84,NJCL-Certamen,1996,NJCL-Certamen_1996_12c_24,short_answer,translation,unknown,english,latin,Say in Latin “twenty-one.”,[],"[VĪGINTĪ ŪNUS, VĪGINTĪ ET ŪNUS]",False,"In Latin, the cardinal number for “twenty-one”...",viginti unus,certamen_translation_long.json,NaN
107,NJCL-Certamen,1996,NJCL-Certamen_1996_20_6,short_answer,translation,unknown,english,latin,Translate the purpose clause into Latin for th...,[],"[UT ŌRĀTIŌNĒS AUDĪRENT, QUAE ŌRĀTIŌNĒS AUDĪREN...",True,"In Latin, the purpose clause “to hear the orat...",ut orationes audirent,certamen_translation_long.json,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
342,latin-grammar-junior-scholarship,1884,latin-grammar-junior-scholarship_113.3.4,short_answer,translation,unknown,english,latin,NaN,[],"[Eum rogarunt utrum, si oppidum cepisset, capt...",False,"Rogaverunt eum, utrum, si oppidum cepisset, ca...","Rogaverunt eum, utrum, si oppidum cepisset, ca...",junior_scholarship_translation_long.json,"Put into Latin: ""They asked him whether, if he..."
343,latin-grammar-junior-scholarship,1884,latin-grammar-junior-scholarship_113.3.5,short_answer,translation,unknown,english,latin,NaN,[],[Promittit se venturum.],False,Ille se venturum esse pollicitur.\n\nAnswer: I...,Ille se venturum esse pollicitur.,junior_scholarship_translation_long.json,"Put into Latin: ""He promises to come."""
344,latin-grammar-junior-scholarship,1884,latin-grammar-junior-scholarship_113.3.6,short_answer,translation,unknown,english,latin,NaN,[],[Sperat se posse venire.],False,Sperat ut possit venire.\n\nAnswer: Sperat ut ...,Sperat ut possit venire.,junior_scholarship_translation_long.json,"Put into Latin: ""He hopes that he will be able..."
346,latin-grammar-junior-scholarship,1884,latin-grammar-junior-scholarship_114.8.2,short_answer,translation,unknown,english,latin,NaN,[],[Extremo tertio libro.],False,Ad finem libri tertii.\n\nAnswer: Ad finem lib...,Ad finem libri tertii.,junior_scholarship_translation_long.json,"Put into Latin—""At the end of the third book."""


In [74]:
lat_const_df

,source_name,source_year,question_id,question_format,question_content,difficulty,question_language,answer_language,question,multiple_choice_options,answers,constrained,raw_resp,parsed_resp,source_file,question_text
107,NJCL-Certamen,1996,NJCL-Certamen_1996_20_6,short_answer,translation,unknown,english,latin,Translate the purpose clause into Latin for th...,[],"[UT ŌRĀTIŌNĒS AUDĪRENT, QUAE ŌRĀTIŌNĒS AUDĪREN...",True,"To translate the purpose clause ""to hear the o...",audire orationes,certamen_translation_long.json,NaN
120,NJCL-Certamen,1996,NJCL-Certamen_1996_16b_35,short_answer,translation,unknown,english,latin,Consider the sentence “The soldiers could run ...,[],[DECEM MĪLIA PASSUUM],True,"To translate ""for ten miles"" into Latin, we ne...",decem milia,certamen_translation_long.json,NaN
136,NJCL-Certamen,1996,NJCL-Certamen_1996_7a_53,short_answer,translation,unknown,english,latin,"Using the verb relinquō, translate into Latin ...",[],[NĒ QUIS URBEM RELINQUERET],True,"To translate the given sentence into Latin, we...","Portae urbis claudebantur, ne nemo urbem relin...",certamen_translation_long.json,NaN
137,NJCL-Certamen,1996,NJCL-Certamen_1996_7b_52,short_answer,translation,unknown,english,latin,"Again, using the verb relinquō, translate into...",[],[UT NĒMŌ URBEM RELĪQUISSET],True,To translate the dependent subjunctive clause ...,Timor omnium hostium tantus erat ut nemo urbem...,certamen_translation_long.json,NaN
153,NJCL-Certamen,1996,NJCL-Certamen_1996_6c_50,short_answer,translation,unknown,english,latin,Say in Latin using the Optative: Would that Ci...,[],"[UTINAM CICERŌ IMPERĀTOR FACTUS ESSET, UTINAM ...",True,Utinam Cicero imperator fieret!\n(Note: The Op...,Utinam Cicero imperator fieret!,certamen_translation_long.json,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
322,latin-grammar-junior-scholarship,1884,latin-grammar-junior-scholarship_102.6.1,short_answer,translation,unknown,english,latin,NaN,[],"[Quum Hannibal profectus est, ver appetebat.]",True,"""Ver cum oriretur, Hannibal proficisceretur"" i...","Cum ver oriretur, Hannibal proficiscebatur.",junior_scholarship_translation_long.json,"Put into Latin using *quum*: ""The spring was c..."
323,latin-grammar-junior-scholarship,1884,latin-grammar-junior-scholarship_102.6.2,short_answer,translation,unknown,english,latin,NaN,[],"[Quum ver appeteret, H. profectus est.]",True,"To express the idea ""Hannibal started when the...",Hannibal proficiscitur quum ver appropinquaret.,junior_scholarship_translation_long.json,"Put into Latin using *quum*: ""Hannibal started..."
324,latin-grammar-junior-scholarship,1884,latin-grammar-junior-scholarship_103.3.2,short_answer,translation,unknown,english,latin,NaN,[],[Mihi obviam ivit.],True,"To express the sentence ""He went to meet me"" i...",Ibat obviam mihi.,junior_scholarship_translation_long.json,"Put into Latin—""He went to meet (*obviam*) me."""
325,latin-grammar-junior-scholarship,1884,latin-grammar-junior-scholarship_103.3.3,short_answer,translation,unknown,english,latin,NaN,[],[Legionibus Caius praefuit.],True,"To translate ""The legions were commanded by Ca...",Legiones a Gaio praebebantur,junior_scholarship_translation_long.json,"Put into Latin—""The legions were commanded (*p..."


In [75]:
lat_const_df.to_csv(f'{model_name}_lat_const.tsv', sep='\t')

In [76]:
lat_unconst_df.to_csv(f'{model_name}_lat_unconst.tsv', sep='\t')

In [52]:
lat_unconst_df

,source_name,source_year,question_id,question_format,question_content,difficulty,question_language,answer_language,question,multiple_choice_options,answers,constrained,raw_resp,parsed_resp,source_file,question_text
19,NJCL-Certamen,1996,NJCL-Certamen_1996_1a_11,short_answer,translation,unknown,english,latin,Say in Latin : Where are the horses of the mas...,[],"[UBI SUNT DOMINĪ CABALLĪ?, UBI SUNT MAGISTRĪ C...",False,"<think>\nOkay, the user wants the Latin transl...",Ubi sunt equi domini?,certamen_translation_long.json,NaN
20,NJCL-Certamen,1996,NJCL-Certamen_1996_1b_11,short_answer,translation,unknown,english,latin,Say in Latin: The boys do not want to stay at ...,[],"[PUERĪ DOMĪ MANĒRE NŌN CUPIUNT, PUERĪ DOMĪ MAN...",False,"<think>\nOkay, the user wants the Latin transl...",Pueri nolunt manēre domī.,certamen_translation_long.json,NaN
21,NJCL-Certamen,1996,NJCL-Certamen_1996_1c_11,short_answer,translation,unknown,english,latin,Say in Latin: My friend was captured by pirates.,[],"[MEUS AMICUS Ā PĪRĀTĪS CAPTUS EST, MEA AMICA Ā...",False,"<think>\nOkay, the user wants the Latin transl...",Amicus meus a piratis captus est.,certamen_translation_long.json,NaN
84,NJCL-Certamen,1996,NJCL-Certamen_1996_12c_24,short_answer,translation,unknown,english,latin,Say in Latin “twenty-one.”,[],"[VĪGINTĪ ŪNUS, VĪGINTĪ ET ŪNUS]",False,"<think>\nOkay, the user is asking for the Lati...",viginti unus,certamen_translation_long.json,NaN
121,NJCL-Certamen,1996,NJCL-Certamen_1996_12c_34,short_answer,translation,unknown,english,latin,Say in Latin “for seven years.”,[],[per septem annōs],False,"<think>\nOkay, the user is asking for ""for sev...",septem annis,certamen_translation_long.json,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
342,latin-grammar-junior-scholarship,1884,latin-grammar-junior-scholarship_113.3.4,short_answer,translation,unknown,english,latin,NaN,[],"[Eum rogarunt utrum, si oppidum cepisset, capt...",False,"<think>\n\nOkay, the user wants to translate t...","Interrogaverunt eum utrum, si oppidum cepisset...",junior_scholarship_translation_long.json,"Put into Latin: ""They asked him whether, if he..."
343,latin-grammar-junior-scholarship,1884,latin-grammar-junior-scholarship_113.3.5,short_answer,translation,unknown,english,latin,NaN,[],[Promittit se venturum.],False,"<think>\nOkay, the user wants ""He promises to ...",Promittit venire.,junior_scholarship_translation_long.json,"Put into Latin: ""He promises to come."""
344,latin-grammar-junior-scholarship,1884,latin-grammar-junior-scholarship_113.3.6,short_answer,translation,unknown,english,latin,NaN,[],[Sperat se posse venire.],False,"<think>\nOkay, the user wants the Latin transl...",Sperat venire posse.,junior_scholarship_translation_long.json,"Put into Latin: ""He hopes that he will be able..."
346,latin-grammar-junior-scholarship,1884,latin-grammar-junior-scholarship_114.8.2,short_answer,translation,unknown,english,latin,NaN,[],[Extremo tertio libro.],False,"<think>\nOkay, the user wants to translate ""At...",In fine libri tertii.,junior_scholarship_translation_long.json,"Put into Latin—""At the end of the third book."""


In [39]:
eng_ans_df

,source_name,source_year,question_id,question_format,question_content,difficulty,question_language,answer_language,question,multiple_choice_options,answers,constrained,raw_resp,parsed_resp,source_file,question_text
0,NJCL-Certamen,1996,NJCL-Certamen_1996_8a_0,short_answer,translation,unknown,english,english,Translate the motto of Alabama: Audēmus iūra n...,[],"[WE DARE TO DEFEND OUR RIGHTS, WE DARE TO DEFE...",False,"<think>\nOkay, the user wants me to translate ...",We dare defend our rights.,certamen_translation_long.json,NaN
1,NJCL-Certamen,1996,NJCL-Certamen_1996_8b_0,short_answer,translation,unknown,english,english,Translate the motto of the Dominion of Canada:...,[],"[FROM SEA TO SEA, , , , , , , , , , , , , , , ...",False,"<think>\nOkay, the user wants me to translate ...",From sea to sea.,certamen_translation_long.json,NaN
2,NJCL-Certamen,1996,NJCL-Certamen_1996_8c_0,short_answer,translation,unknown,english,english,Translate the motto of Wellesley College: nōn ...,[],"[NOT TO BE SERVED, NOT TO BE MANAGED, BUT TO S...",False,"<think>\nOkay, the user wants me to translate ...","Not to be served, but to serve.",certamen_translation_long.json,NaN
3,NJCL-Certamen,1996,NJCL-Certamen_1996_15a_3,short_answer,translation,unknown,english,english,Translate into English the following maxim of ...,[],"[If you want peace, prepare for war, , , , , ,...",False,"<think>\nOkay, the user wants the translation ...","If you wish for peace, prepare for war.",certamen_translation_long.json,NaN
4,NJCL-Certamen,1996,NJCL-Certamen_1996_15b_3,short_answer,translation,unknown,english,english,Translate into English the following quotation...,[],"[Divine nature gave us the fields, human art b...",False,"<think>\nOkay, the user wants me to translate ...",Divine Nature gave the fields; human art built...,certamen_translation_long.json,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
337,latin-grammar-junior-scholarship,1884,latin-grammar-junior-scholarship_109.9.2,short_answer,translation,unknown,english,english,NaN,[],"[Will thirst for honours., , , , , , , , , , ,...",False,"<think>\nOkay, the user wants me to translate ...",He will thirst for honors.,junior_scholarship_translation_long.json,Translate—*Honores sitiet.*
338,latin-grammar-junior-scholarship,1884,latin-grammar-junior-scholarship_109.9.3,short_answer,translation,unknown,english,english,NaN,[],"[The poor hungry Greek, if you bid him, will g...",False,"<think>\nOkay, I need to translate the Latin s...",A hungry little Greek will go to heaven if you...,junior_scholarship_translation_long.json,"Translate—*Graeculus esuriens in coelum, jusse..."
345,latin-grammar-junior-scholarship,1884,latin-grammar-junior-scholarship_114.7.1,short_answer,translation,unknown,english,english,NaN,[],"[The more a man denies himself, the more will ...",True,"<think>\nOkay, so the user wants me to transla...","The more one denies oneself, the more one will...",junior_scholarship_translation_long.json,Translate the following Latin phrase into Engl...
348,latin-grammar-junior-scholarship,1884,latin-grammar-junior-scholarship_app.V.4.1,short_answer,translation,unknown,english,english,NaN,[],"[Let war yield to peace., , , , , , , , , , , ...",False,"<think>\nOkay, the user wants me to translate ...",Let military power yield to civilian authority.,junior_scholarship_translation_long.json,Translate —Cedant arma togae.


In [50]:
eng_const_df

,source_name,source_year,question_id,question_format,question_content,difficulty,question_language,answer_language,question,multiple_choice_options,answers,constrained,raw_resp,parsed_resp,source_file,question_text
62,NJCL-Certamen,1996,NJCL-Certamen_1996_12b_19,short_answer,translation,unknown,english,english,Translate nūbēs as a verb.,[],"[YOU WILL MARRY, , , , , , , , , , , , , , , ,...",True,"<think>\nOkay, the user wants me to translate ...",he/she/it marries (third person singular prese...,certamen_translation_long.json,NaN
67,NJCL-Certamen,1996,NJCL-Certamen_1996_8b_11,short_answer,translation,unknown,english,english,Translate potior as a verb.,[],"[GAIN POSSESSION OF, BE POWERFUL OVER, , , , ,...",True,"<think>\nOkay, the user is asking me to transl...","potior (deponent) means ""to take possession of...",certamen_translation_long.json,NaN
262,NJCL-Certamen,2000,NJCL-Certamen_2000_18a_10,short_answer,translation,unknown,english,english,Translate the following sentence into English ...,[],"[AFTER HE TASTED THE FOOD, THE MASTER PRAISED ...",True,"<think>\nOkay, the user wants me to translate ...","The lord, after he tasted the food, praised th...",certamen_translation_long.json,NaN
263,NJCL-Certamen,2000,NJCL-Certamen_2000_18b_10,short_answer,translation,unknown,english,english,Translate the following sentence into English ...,[],"[WHILE THE DOG WAS SLEEPING, THE CAT ATE THE B...",True,"<think>\nOkay, so the user wants me to transla...","While the dog is sleeping, the cat has consume...",certamen_translation_long.json,NaN
264,NJCL-Certamen,2000,NJCL-Certamen_2000_18c_9,short_answer,translation,unknown,english,english,Translate the following sentence into English ...,[],"[WHEN I SEE MY FATHER, I WILL ASK HIM FOR MONE...",True,"<think>\nOkay, let's tackle this translation. ...","When I see my father, I will ask him money.",certamen_translation_long.json,NaN
343,NJCL-Certamen,2000,NJCL-Certamen_2000_20b_23,short_answer,translation,unknown,english,english,Translate the following sentence into idiomati...,[],"[IF CICERO WEREN’T A GOOD ORATOR, THE JUDGE WO...",True,"<think>\n\nOkay, so the user wants me to trans...","Unless Cicero were a good orator, the judge wo...",certamen_translation_long.json,NaN
429,NJCL-Certamen,2002,NJCL-Certamen_2002_6b_21,short_answer,translation,unknown,english,english,Minor nātū is an idiom. Translate that phrase ...,[],"[SMALLER BY BIRTH, LESSER BY BIRTH, , , , , , ...",True,"<think>\nOkay, the user is asking about the li...",younger by birth,certamen_translation_long.json,NaN
450,NJCL-Certamen,2002,NJCL-Certamen_2002_14b_24,short_answer,translation,unknown,english,english,Translate that participle.,[],"[HAVING LEFT, STARTED OUT, , , , , , , , , , ,...",True,"<think>\n\nOkay, the user asked me to translat...",Please provide the specific participle you wou...,certamen_translation_long.json,NaN
551,NJCL-Certamen,2002,NJCL-Certamen_2002_5a_41,short_answer,translation,unknown,english,english,What is the literal translation of the charact...,[],"[THE SORTING HAT, THE DISTRIBUTING HAT, , , , ...",True,"<think>\nOkay, let's see. The user is asking f...","Petasus Distribuens literally translates to ""P...",certamen_translation_long.json,NaN
629,NJCL-Certamen,2002,NJCL-Certamen_2002_5_24,short_answer,translation,unknown,english,english,"The Lexicon Recentis Latīnitātis, or Dictionar...",[],"[THE EQUALING OF ALL WEALTH, THE LEVELING OF A...",True,"<think>\nOkay, the user wants me to translate ...",the equalization of all goods,certamen_translation_long.json,NaN


In [53]:
eng_unconst_df

,source_name,source_year,question_id,question_format,question_content,difficulty,question_language,answer_language,question,multiple_choice_options,answers,constrained,raw_resp,parsed_resp,source_file,question_text
0,NJCL-Certamen,1996,NJCL-Certamen_1996_8a_0,short_answer,translation,unknown,english,english,Translate the motto of Alabama: Audēmus iūra n...,[],"[WE DARE TO DEFEND OUR RIGHTS, WE DARE TO DEFE...",False,"<think>\nOkay, the user wants me to translate ...",We dare defend our rights.,certamen_translation_long.json,NaN
1,NJCL-Certamen,1996,NJCL-Certamen_1996_8b_0,short_answer,translation,unknown,english,english,Translate the motto of the Dominion of Canada:...,[],"[FROM SEA TO SEA, , , , , , , , , , , , , , , ...",False,"<think>\nOkay, the user wants me to translate ...",From sea to sea.,certamen_translation_long.json,NaN
2,NJCL-Certamen,1996,NJCL-Certamen_1996_8c_0,short_answer,translation,unknown,english,english,Translate the motto of Wellesley College: nōn ...,[],"[NOT TO BE SERVED, NOT TO BE MANAGED, BUT TO S...",False,"<think>\nOkay, the user wants me to translate ...","Not to be served, but to serve.",certamen_translation_long.json,NaN
3,NJCL-Certamen,1996,NJCL-Certamen_1996_15a_3,short_answer,translation,unknown,english,english,Translate into English the following maxim of ...,[],"[If you want peace, prepare for war, , , , , ,...",False,"<think>\nOkay, the user wants the translation ...","If you wish for peace, prepare for war.",certamen_translation_long.json,NaN
4,NJCL-Certamen,1996,NJCL-Certamen_1996_15b_3,short_answer,translation,unknown,english,english,Translate into English the following quotation...,[],"[Divine nature gave us the fields, human art b...",False,"<think>\nOkay, the user wants me to translate ...",Divine Nature gave the fields; human art built...,certamen_translation_long.json,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
336,latin-grammar-junior-scholarship,1884,latin-grammar-junior-scholarship_109.9.1,short_answer,translation,unknown,english,english,NaN,[],[Respected the oaths and vow of the suppliant....,False,"<think>\n\nOkay, so I need to translate the La...",He was ashamed of the suppliant's laws and faith.,junior_scholarship_translation_long.json,Translate—*Jura fidemque Supplicis erubuit.*
337,latin-grammar-junior-scholarship,1884,latin-grammar-junior-scholarship_109.9.2,short_answer,translation,unknown,english,english,NaN,[],"[Will thirst for honours., , , , , , , , , , ,...",False,"<think>\nOkay, the user wants me to translate ...",He will thirst for honors.,junior_scholarship_translation_long.json,Translate—*Honores sitiet.*
338,latin-grammar-junior-scholarship,1884,latin-grammar-junior-scholarship_109.9.3,short_answer,translation,unknown,english,english,NaN,[],"[The poor hungry Greek, if you bid him, will g...",False,"<think>\nOkay, I need to translate the Latin s...",A hungry little Greek will go to heaven if you...,junior_scholarship_translation_long.json,"Translate—*Graeculus esuriens in coelum, jusse..."
348,latin-grammar-junior-scholarship,1884,latin-grammar-junior-scholarship_app.V.4.1,short_answer,translation,unknown,english,english,NaN,[],"[Let war yield to peace., , , , , , , , , , , ...",False,"<think>\nOkay, the user wants me to translate ...",Let military power yield to civilian authority.,junior_scholarship_translation_long.json,Translate —Cedant arma togae.


In [40]:
# avg amount of Latin references
all_lat_refs = latin_ans_df['answers'].to_list()
all_eng_refs = eng_ans_df['answers'].to_list()

In [41]:
import numpy as np

In [42]:
# mean, median, min, max
lat_lens = [len(r) for r in all_lat_refs]
eng_lens = [len(r) for r in all_eng_refs]

lat_mean = np.mean(lat_lens)
lat_med = np.median(lat_lens)
lat_min = np.min(lat_lens)
lat_max = np.max(lat_lens)

eng_mean = np.mean(eng_lens)
eng_med = np.median(eng_lens)
eng_min = np.min(eng_lens)
eng_max = np.max(eng_lens)

print('lat min:', lat_min)
print('lat mean:', lat_mean)
print('lat med:', lat_med)
print('lat max:', lat_max)

print('eng min:', eng_min)
print('eng mean:', eng_mean)
print('eng med:', eng_med)
print('eng max:', eng_max)

lat min: 1
lat mean: 2.2134570765661254
lat med: 1.0
lat max: 30
eng min: 48
eng mean: 48.0
eng med: 48.0
eng max: 48


In [43]:
eng_lens

[48,
 48,
 48,
 48,
 48,
 48,
 48,
 48,
 48,
 48,
 48,
 48,
 48,
 48,
 48,
 48,
 48,
 48,
 48,
 48,
 48,
 48,
 48,
 48,
 48,
 48,
 48,
 48,
 48,
 48,
 48,
 48,
 48,
 48,
 48,
 48,
 48,
 48,
 48,
 48,
 48,
 48,
 48,
 48,
 48,
 48,
 48,
 48,
 48,
 48,
 48,
 48,
 48,
 48,
 48,
 48,
 48,
 48,
 48,
 48,
 48,
 48,
 48,
 48,
 48,
 48,
 48,
 48,
 48,
 48,
 48,
 48,
 48,
 48,
 48,
 48,
 48,
 48,
 48,
 48,
 48,
 48,
 48,
 48,
 48,
 48,
 48,
 48,
 48,
 48,
 48,
 48,
 48,
 48,
 48,
 48,
 48,
 48,
 48,
 48,
 48,
 48,
 48,
 48,
 48,
 48,
 48,
 48,
 48,
 48,
 48,
 48,
 48,
 48,
 48,
 48,
 48,
 48,
 48,
 48,
 48,
 48,
 48,
 48,
 48,
 48,
 48,
 48,
 48,
 48,
 48,
 48,
 48,
 48,
 48,
 48,
 48,
 48,
 48,
 48,
 48,
 48,
 48,
 48,
 48,
 48,
 48,
 48,
 48,
 48,
 48,
 48,
 48,
 48,
 48,
 48,
 48,
 48,
 48,
 48,
 48,
 48,
 48,
 48,
 48,
 48,
 48,
 48,
 48,
 48,
 48,
 48,
 48,
 48,
 48,
 48,
 48,
 48,
 48,
 48,
 48,
 48,
 48,
 48,
 48,
 48,
 48,
 48,
 48,
 48,
 48,
 48,
 48,
 48,
 48,
 48,
 48,
 48,
 48,
 48,


In [18]:
from sacrebleu.metrics import BLEU

bleu = BLEU(lowercase=True)

overall 

In [19]:
bleu.corpus_score(lat_sys, lat_refs)

BLEU = 11.12 39.2/15.5/7.3/3.5 (BP = 1.000 ratio = 1.215 hyp_len = 2664 ref_len = 2192)

In [20]:
bleu.corpus_score(lat_sys_no_macrons, lat_refs_no_macrons)

BLEU = 24.67 56.6/30.0/18.7/11.7 (BP = 1.000 ratio = 1.216 hyp_len = 2666 ref_len = 2192)

In [21]:
bleu.corpus_score(eng_sys, eng_refs)

BLEU = 43.14 66.7/48.4/37.1/28.9 (BP = 1.000 ratio = 1.110 hyp_len = 8016 ref_len = 7220)

constrained vs unconstrained

In [22]:
# target latin
print('with macrons:')
print('constrained:', bleu.corpus_score(lat_const_sys, lat_const_refs))
print('unconstrained:', bleu.corpus_score(lat_unconst_sys, lat_unconst_refs))

print()
print('without macrons:')
print('constrained:', bleu.corpus_score(lat_const_sys_no_mac, lat_const_refs_no_mac))
print('unconstrained:', bleu.corpus_score(lat_unconst_sys_no_mac, lat_unconst_refs_no_mac))


with macrons:
constrained: BLEU = 9.85 39.8/15.4/6.8/2.3 (BP = 1.000 ratio = 1.170 hyp_len = 880 ref_len = 752)
unconstrained: BLEU = 11.63 38.8/15.5/7.5/4.1 (BP = 1.000 ratio = 1.239 hyp_len = 1784 ref_len = 1440)

without macrons:
constrained: BLEU = 27.25 59.7/32.8/21.1/13.3 (BP = 1.000 ratio = 1.170 hyp_len = 880 ref_len = 752)
unconstrained: BLEU = 23.42 55.1/28.6/17.5/10.9 (BP = 1.000 ratio = 1.240 hyp_len = 1786 ref_len = 1440)


In [23]:
# target english
print('constrained:', bleu.corpus_score(eng_const_sys, eng_const_refs))
print('unconstrained:', bleu.corpus_score(eng_unconst_sys, eng_unconst_refs))

constrained: BLEU = 34.68 62.6/43.1/29.2/18.4 (BP = 1.000 ratio = 1.081 hyp_len = 147 ref_len = 136)
unconstrained: BLEU = 43.29 66.8/48.5/37.2/29.1 (BP = 1.000 ratio = 1.111 hyp_len = 7869 ref_len = 7084)
